# Label-Agnostic SATURN: Human, Monkey, Mouse, Frog and Fish (PyTorch)

This notebook presents the complete benchmark cell by cell: data inspection, shared SATURN pretraining, the labeled baseline, construction and auditing of the label-free training artifact, InfoNCE, MMD, partial OT, post-hoc evaluation, and the final report figures.

The default mode is a fresh walkthrough run into `out/human_monkey_mouse_frog_fish_pytorch_benchmark_walkthrough`. After one completed run, set `HMMFF_REPORT_SOURCE_DIR` to that output directory and run section 6 to regenerate figures from saved embeddings and metrics without training again.

Based on the pre-JAX walkthrough from Git commit `27ed84d`, using the original PyTorch trainers and `.pt` checkpoints. This copy includes frog (`x_laevis`) and zebrafish (`d_rerio`) alongside the three mammals, with 20 shared pretraining epochs and 5 fine-tuning epochs for every trial. Broad-category ontologies are currently available only for the three mammals; frog and fish remain `unknown` in broad-category plots and are excluded from broad-category preservation scores. Their fine labels are retained throughout evaluation.


## Experiment design

All four trials begin from the same pretrained SATURN checkpoint.

- **Baseline:** SATURN's original label-aware triplet fine-tuning.
- **InfoNCE:** reciprocal cross-species neighbours inferred from pretrained embeddings and macrogenes.
- **MMD:** pairwise multi-species distribution matching.
- **OT:** entropy-regularized partial optimal transport.

The three label-free trainers can access only embeddings, macrogenes, species identity, and observation IDs. Cell labels are introduced only after training by the shared evaluator.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
import time

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import torch
from IPython.display import Markdown, display

repo_root = next((
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'scripts' / 'human_monkey_mouse.sh').exists()
), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate the macrogenes repository.')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Fresh rerun mode: execute training stages into the walkthrough output directory.
# Set to False only when intentionally inspecting already-completed artifacts.
REBUILD_FROM_SCRATCH = True

canonical_output_dir = repo_root / 'out' / 'human_monkey_mouse_frog_fish_pytorch_benchmark_clean'
fresh_output_dir = repo_root / 'out' / 'human_monkey_mouse_frog_fish_pytorch_benchmark_walkthrough'
output_dir = fresh_output_dir

SEED = 0
DEVICE = 'cuda'
DEVICE_NUM = 0
PRETRAIN_EPOCHS = 20
METRIC_EPOCHS = 5
BATCH_SIZE = 512
TRIALS = ['baseline', 'infonce', 'mmd', 'ot']

if REBUILD_FROM_SCRATCH and output_dir == canonical_output_dir:
    raise RuntimeError('Fresh runs must not overwrite the canonical presentation outputs.')
if REBUILD_FROM_SCRATCH and not torch.cuda.is_available():
    raise RuntimeError('A system GPU is required for a fresh run.')

shared_dir = output_dir / 'shared'
baseline_dir = output_dir / 'baseline'
pretrain_model = shared_dir / 'pretrain_model.pt'
centroids = shared_dir / f'centroids_seed{SEED}.pkl'
baseline_model = baseline_dir / 'final_model.pt'
pretrain_adata = baseline_dir / 'saturn_results' / 'adata_pretrain.h5ad'
baseline_adata = baseline_dir / 'saturn_results' / 'final_adata.h5ad'
label_free_artifact = shared_dir / 'label_free_artifact.npz'
evaluation_triplets = shared_dir / 'evaluation_triplets.npz'
artifact_metadata = shared_dir / 'artifact_metadata.json'

shared_dir.mkdir(parents=True, exist_ok=True)
baseline_dir.mkdir(parents=True, exist_ok=True)

base_env = os.environ.copy()
base_env.update({
    'PYTHONHASHSEED': str(SEED),
    'NUMBA_NUM_THREADS': '1',
    'NUMBA_CACHE_DIR': '/tmp/macrogenes-numba-cache',
    'MPLCONFIGDIR': '/tmp/macrogenes-matplotlib',
})

def make_label_palette(labels):
    label_index = pd.Index(pd.Series(labels).astype(str).dropna().unique()).sort_values()
    colors = sns.color_palette('husl', n_colors=len(label_index)).as_hex()
    return dict(zip(label_index, colors))

def run_stage(name, command, required_outputs, env=None):
    required_outputs = [Path(path) for path in required_outputs]
    if not REBUILD_FROM_SCRATCH and all(path.exists() for path in required_outputs):
        print(f'✓ {name}: reusing completed outputs')
        for path in required_outputs:
            print(f'  {path.relative_to(repo_root)}')
        return
    print(f'▶ {name}')
    print('  $', shlex.join([str(part) for part in command]))
    started = time.time()
    subprocess.run(
        [str(part) for part in command],
        cwd=repo_root,
        env=env or base_env,
        check=True,
    )
    missing = [path for path in required_outputs if not path.exists()]
    if missing:
        raise FileNotFoundError(f'{name} did not create: {missing}')
    print(f'✓ {name}: completed in {(time.time() - started) / 60:.1f} minutes')

print(f'Repository: {repo_root}')
print(f'Python environment: {sys.prefix}')
print(f'Output directory: {output_dir}')
print(f'Mode: {"fresh GPU run" if REBUILD_FROM_SCRATCH else "completed-output inspection"}')
print(f'Metric-learning epochs: {METRIC_EPOCHS}')

> **Run note:** `REBUILD_FROM_SCRATCH = True` executes the training and evaluation stages into the walkthrough output directory. To regenerate figures only, keep the completed artifacts in place, set `HMMFF_REPORT_SOURCE_DIR` to that run directory, and run the setup cell plus section 6.


## 1. Inspect the benchmark data

The benchmark includes human (`h_sapiens`), mouse (`m_musculus`), mouse lemur (`m_murinus`), frog (`x_laevis`), and zebrafish (`d_rerio`). Cell counts are computed from the input atlases below. Labels are read here for description and later evaluation, but the label-free fine-tuning artifact deliberately excludes them.

In [ ]:
input_table = pd.read_csv(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv')
display(input_table)

dataset_rows = []
for row in input_table.itertuples(index=False):
    atlas = ad.read_h5ad(repo_root / row.path, backed='r')
    label_column = row.in_label_col
    dataset_rows.append({
        'species': row.species,
        'cells': atlas.n_obs,
        'genes': atlas.n_vars,
        'label_column': label_column,
        'unique_labels': int(atlas.obs[label_column].nunique()),
    })
    atlas.file.close()

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)
print(f'Total cells: {dataset_summary.cells.sum():,}')

## 2. Shared pretraining and labeled SATURN baseline

This stage performs SATURN pretraining once, saves the shared checkpoint, and then runs the original label-aware metric-learning baseline for 5 epochs.


In [ ]:
baseline_env = base_env.copy()
baseline_env.update({
    'IN_DATA': str(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv'),
    'PRETRAIN_EPOCHS': str(PRETRAIN_EPOCHS),
    'SEED': str(SEED),
    'DEVICE': DEVICE,
    'DEVICE_NUM': str(DEVICE_NUM),
    'WORK_DIR': str(baseline_dir) + '/',
    'CENTROIDS_INIT_PATH': str(centroids),
    'PRETRAIN_MODEL_PATH': str(pretrain_model),
    'METRIC_MODEL_PATH': str(baseline_model),
    'PRETRAIN': 'true',
    'METRIC_EPOCHS': str(METRIC_EPOCHS),
    'METRIC_BATCH_SIZE': str(BATCH_SIZE),
    'POLLING_FREQ': '5',
})

run_stage(
    'shared pretraining + labeled baseline',
    ['bash', repo_root / 'scripts' / 'human_monkey_mouse.sh'],
    [pretrain_model, pretrain_adata, baseline_model, baseline_adata],
    env=baseline_env,
)

In [ ]:
pretrained = ad.read_h5ad(pretrain_adata, backed='r')
baseline = ad.read_h5ad(baseline_adata, backed='r')
display(pd.DataFrame([
    {'stage': 'pretrained', 'cells': pretrained.n_obs, 'embedding_dim': pretrained.n_vars, 'macrogenes': pretrained.obsm['macrogenes'].shape[1]},
    {'stage': 'labeled baseline', 'cells': baseline.n_obs, 'embedding_dim': baseline.n_vars, 'macrogenes': baseline.obsm['macrogenes'].shape[1]},
]))
pretrained.file.close()
baseline.file.close()

### Pretrained representation before fine-tuning

This is the visual starting point for all four trials. Species are initially almost completely separated, while much of the within-species cell-state structure is already present.

In [ ]:
pretrained = ad.read_h5ad(pretrain_adata)
if 'X_umap' not in pretrained.obsm:
    sc.pp.neighbors(pretrained, n_neighbors=15, metric='cosine', random_state=SEED)
    sc.tl.umap(pretrained, min_dist=0.3, random_state=SEED)

def make_label_palette(labels):
    label_index = pd.Index(pd.Series(labels).astype(str).dropna().unique()).sort_values()
    colors = sns.color_palette('husl', n_colors=len(label_index)).as_hex()
    return dict(zip(label_index, colors))

label_palette = make_label_palette(pretrained.obs['labels2'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.umap(pretrained, color='species', ax=axes[0], show=False, title='Pretrained: species', size=8)
sc.pl.umap(pretrained, color='labels2', ax=axes[1], show=False, title='Pretrained: labels', size=8, legend_loc=None, palette=label_palette)
fig.suptitle('Shared pretrained SATURN representation', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
from label_agnostic.metrics import embedding_benchmark_metrics

pretrained_metrics = embedding_benchmark_metrics(
    np.asarray(pretrained.X),
    pretrained.obs['species'].astype(str).to_numpy(),
    pretrained.obs['labels2'].astype(str).to_numpy(),
    k=15,
)
display(pd.Series(pretrained_metrics, name='pretrained'))

## 3. Build and audit the label-free training artifact

The exporter creates two separate files:

1. A strict training artifact with no label arrays.
2. A frozen label-aware triplet manifest used only by the evaluator after all training is complete.

In [ ]:
run_stage(
    'label-free artifact + frozen evaluation triplets',
    [
        sys.executable,
        repo_root / 'scripts' / 'prepare_label_agnostic_artifacts.py',
        '--pretrain-adata', pretrain_adata,
        '--artifact', label_free_artifact,
        '--triplets', evaluation_triplets,
        '--metadata', artifact_metadata,
        '--seed', str(SEED),
        '--batch-size', str(BATCH_SIZE),
    ],
    [label_free_artifact, evaluation_triplets, artifact_metadata],
)

In [ ]:
training_artifact = np.load(label_free_artifact, allow_pickle=False)
allowed_keys = {'embeddings', 'macrogenes', 'species', 'obs_ids'}
actual_keys = set(training_artifact.files)
assert actual_keys == allowed_keys
assert not any('label' in key.lower() for key in actual_keys)

artifact_audit = pd.DataFrame([
    {'array': key, 'shape': training_artifact[key].shape, 'dtype': str(training_artifact[key].dtype)}
    for key in training_artifact.files
])
display(artifact_audit)
display(json.loads(artifact_metadata.read_text()))
print('✓ No cell-label field is available to label-free fine-tuning.')

## 4. Run the three label-free objectives

Each trial loads the same four-field artifact and the same pretrained checkpoint. Checkpoint selection uses species mixing subject to a frozen-neighbour recall constraint; labels are never used for optimization or selection.

In [ ]:
def run_label_free_trial(objective):
    trial_dir = output_dir / objective
    command = [
        sys.executable,
        repo_root / 'scripts' / 'train_label_agnostic.py',
        '--objective', objective,
        '--artifact', label_free_artifact,
        '--pretrain-checkpoint', pretrain_model,
        '--output-dir', trial_dir,
        '--device', DEVICE,
        '--device-num', str(DEVICE_NUM),
        '--seed', str(SEED),
        '--epochs', str(METRIC_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
    ]
    run_stage(
        f'{objective} fine-tuning',
        command,
        [trial_dir / 'final_embeddings.npz', trial_dir / 'metric_history.csv', trial_dir / 'run_summary.json'],
    )
    return json.loads((trial_dir / 'run_summary.json').read_text())

def show_trial_summary(summary):
    display(pd.Series({
        'objective': summary['objective'],
        'label_free': summary['label_free'],
        'artifact keys seen': ', '.join(summary['artifact_keys_seen_by_trainer']),
        'selected epoch': summary['selected_epoch'],
        'selected species mixing': summary['selected_species_mixing_fraction'],
        'selected frozen-neighbour recall@50': summary['selected_teacher_top15_recall_at_50'],
        'selection uses labels': summary['selection_uses_labels'],
    }).to_frame('value'))

### 4a. InfoNCE

Reciprocal cross-species neighbours inferred from the pretrained embedding/macrogene views serve as positives. Cells from the target species form the contrastive denominator.

In [ ]:
infonce_summary = run_label_free_trial('infonce')
show_trial_summary(infonce_summary)

### 4b. MMD

A multi-bandwidth RBF maximum mean discrepancy is averaged across every species pair.

In [ ]:
mmd_summary = run_label_free_trial('mmd')
show_trial_summary(mmd_summary)

### 4c. Partial optimal transport

A frozen cost based on pretrained embeddings and macrogenes defines an entropy-regularized partial transport plan. The student minimizes the transported embedding cost.

In [ ]:
ot_summary = run_label_free_trial('ot')
show_trial_summary(ot_summary)

## 5. Shared post-hoc evaluation

Only now are labels joined back by observation ID. Every trial receives the same cosine 15-neighbour metrics, frozen triplet-margin evaluation, deterministic UMAP settings, and output schema.

In [ ]:
run_stage(
    'shared post-hoc evaluation',
    [
        sys.executable,
        repo_root / 'scripts' / 'evaluate_label_agnostic_benchmark.py',
        '--root', output_dir,
        '--truth-adata', pretrain_adata,
        '--triplets', evaluation_triplets,
        '--seed', str(SEED),
    ],
    [
        output_dir / 'comparison.csv',
        output_dir / 'acceptance.json',
        *[output_dir / trial / 'evaluated_adata.h5ad' for trial in TRIALS],
    ],
)

## 6. Results and report figures

The remaining cells load a completed benchmark run through `figure_source_dir`. By default this is the walkthrough output directory; set `HMMFF_REPORT_SOURCE_DIR` to another completed run to regenerate the report figures from saved embeddings, UMAP coordinates, histories, and metric summaries without training again.


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import yaml

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from label_agnostic.metrics import joint_neighbors

figure_source_dir = Path(os.environ.get('HMMFF_REPORT_SOURCE_DIR', output_dir)).expanduser()
if not figure_source_dir.is_absolute():
    figure_source_dir = repo_root / figure_source_dir
figure_source_dir = figure_source_dir.resolve()
assets = figure_source_dir / 'report_assets'
assets.mkdir(parents=True, exist_ok=True)

TRIAL_DISPLAY = {
    'baseline': 'SATURN baseline',
    'infonce': 'InfoNCE',
    'mmd': 'MMD-MA',
    'ot': 'Partial OT',
}
LABEL_FREE_TRIALS = [trial for trial in TRIALS if trial != 'baseline']
TRIAL_PALETTE = dict(zip(TRIALS, sns.color_palette('Set2', n_colors=len(TRIALS)).as_hex()))

macro_category_palette = {
    'immune': '#4C78A8',
    'epithelial': '#F58518',
    'endothelial': '#4FA146',
    'mesenchymal': '#B279A2',
    'other': '#8C8C8C',
    'unknown': '#D0D0D0',
}

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def relative_report_path(path):
    path = Path(path)
    try:
        return path.relative_to(repo_root).as_posix()
    except ValueError:
        return str(path)

def save_report_figure(fig, stem, dpi=300, bbox_inches='tight'):
    saved = []
    for suffix in ('png', 'pdf'):
        path = assets / f'{stem}.{suffix}'
        save_kwargs = {'dpi': dpi, 'bbox_inches': bbox_inches}
        if suffix == 'pdf':
            save_kwargs['metadata'] = {'Creator': 'human_monkey_mouse_frog_fish.ipynb'}
        fig.savefig(path, **save_kwargs)
        saved.append(path)
    print('Saved ' + ', '.join(relative_report_path(path) for path in saved))
    return saved

def load_trial_adata(trial):
    path = figure_source_dir / trial / 'evaluated_adata.h5ad'
    if not path.exists():
        raise FileNotFoundError(
            f'Missing evaluated AnnData for {trial}: {path}. '
            'Run the benchmark or set HMMFF_REPORT_SOURCE_DIR to a completed run.'
        )
    return sc.read_h5ad(path)

def load_macro_category_maps():
    local_table = pd.read_csv(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv')
    species_maps = {}
    category_order = None
    for species in local_table['species'].astype(str):
        ontology_path = repo_root / 'data' / 'cell_type_ontology' / f'{species}.yml'
        if not ontology_path.exists():
            print(f'No broad-category ontology for {species}; retaining fine labels and marking macro categories unknown.')
            species_maps[species] = {}
            continue
        with ontology_path.open() as handle:
            annotations = yaml.safe_load(handle)['cell_annotations']
        species_order = [str(category) for category in annotations['supertype_order']]
        if category_order is None:
            category_order = species_order
        elif species_order != category_order:
            raise ValueError(f'Ontology supertype order differs for {species}: {species_order}')
        label_to_category = {}
        for category in species_order:
            for label in annotations['cell_supertypes'][category]:
                label_to_category[str(label)] = str(category)
        species_maps[species] = label_to_category
    return species_maps, [*(category_order or []), 'unknown']

species_to_macro_category, macro_category_order = load_macro_category_maps()

def macro_categories_for(species_values, label_values):
    return np.asarray([
        species_to_macro_category.get(str(species), {}).get(str(label), 'unknown')
        for species, label in zip(species_values, label_values)
    ], dtype=object)

def add_macro_categories(obs_frame):
    result = obs_frame.copy()
    result['species'] = result['species'].astype(str)
    result['labels2'] = result['labels2'].astype(str)
    result['macro_category'] = macro_categories_for(result['species'], result['labels2'])
    return result

def load_report_truth_obs():
    pretrain_path = figure_source_dir / 'baseline' / 'saturn_results' / 'adata_pretrain.h5ad'
    if pretrain_path.exists():
        truth = ad.read_h5ad(pretrain_path, backed='r')
        obs = truth.obs[['species', 'labels2']].copy()
        truth.file.close()
        return add_macro_categories(obs)

    local_table = pd.read_csv(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv')
    rows = []
    for row in local_table.itertuples(index=False):
        atlas = ad.read_h5ad(repo_root / row.path, backed='r')
        rows.append(pd.DataFrame({
            'species': row.species,
            'labels2': atlas.obs[row.in_label_col].astype(str).to_numpy(),
        }))
        atlas.file.close()
    return add_macro_categories(pd.concat(rows, ignore_index=True))

_neighbor_diagnostic_cache = {}

def compute_neighbor_diagnostics(trial, k=15):
    cache_key = (trial, k)
    if cache_key in _neighbor_diagnostic_cache:
        return _neighbor_diagnostic_cache[cache_key]

    trial_adata = load_trial_adata(trial)
    embeddings = np.asarray(trial_adata.X, dtype=np.float32)
    species = trial_adata.obs['species'].astype(str).to_numpy()
    labels = trial_adata.obs['labels2'].astype(str).to_numpy()
    macros = macro_categories_for(species, labels)
    neighbors = joint_neighbors(embeddings, k=k)

    neighbor_species = species[neighbors]
    neighbor_labels = labels[neighbors]
    neighbor_macros = macros[neighbors]
    cell_frame = pd.DataFrame({
        'trial': trial,
        'species': species,
        'labels2': labels,
        'macro_category': macros,
        'cross_species_neighbor_fraction': np.mean(neighbor_species != species[:, None], axis=1),
        'same_label_neighbor_fraction': np.mean(neighbor_labels == labels[:, None], axis=1),
        'same_macro_neighbor_fraction': np.where(
            macros != 'unknown',
            np.mean(neighbor_macros == macros[:, None], axis=1),
            np.nan,
        ),
    })

    cross_summary = (
        cell_frame
        .groupby('species', observed=True)['cross_species_neighbor_fraction']
        .mean()
        .reset_index()
        .rename(columns={'species': 'anchor_species'})
    )
    cross_summary['trial'] = trial

    preservation_summary = (
        cell_frame
        .groupby(['species', 'macro_category'], observed=True)
        .agg(
            cells=('labels2', 'size'),
            same_macro_neighbor_fraction=('same_macro_neighbor_fraction', 'mean'),
            same_label_neighbor_fraction=('same_label_neighbor_fraction', 'mean'),
        )
        .reset_index()
    )
    preservation_summary['trial'] = trial

    result = (cell_frame, cross_summary, preservation_summary)
    _neighbor_diagnostic_cache[cache_key] = result
    return result

print(f'Report figure source: {relative_report_path(figure_source_dir)}')
print(f'Report assets: {relative_report_path(assets)}')


In [ ]:
summary = pd.read_csv(figure_source_dir / 'comparison.csv')
acceptance = json.loads((figure_source_dir / 'acceptance.json').read_text())

summary['method'] = summary['trial'].map(TRIAL_DISPLAY)
display(summary)
display(acceptance)


In [ ]:
expected_epochs = list(range(1, METRIC_EPOCHS + 1))
epoch_failures = {}
for trial in TRIALS:
    history = pd.read_csv(figure_source_dir / trial / 'history.csv')
    observed_epochs = history['epoch'].astype(int).tolist()
    if observed_epochs != expected_epochs:
        epoch_failures[trial] = observed_epochs
if epoch_failures:
    raise AssertionError(f'Unexpected benchmark epoch histories: {epoch_failures}')

baseline_history = pd.read_csv(figure_source_dir / 'baseline' / 'saturn_results' / 'metric_history.csv')
baseline_epochs = baseline_history['epoch'].astype(int).tolist()
if baseline_epochs != expected_epochs:
    raise AssertionError(f'Unexpected baseline metric epochs: {baseline_epochs}')

for objective in TRIALS:
    if objective == 'baseline':
        continue
    trial_summary = json.loads((figure_source_dir / objective / 'run_summary.json').read_text())
    if int(trial_summary['epochs']) != METRIC_EPOCHS:
        raise AssertionError(f'{objective} recorded epochs={trial_summary["epochs"]}, expected {METRIC_EPOCHS}')
    if int(trial_summary['seed']) != SEED:
        raise AssertionError(f'{objective} recorded seed={trial_summary["seed"]}, expected {SEED}')

print(f'OK benchmark verification: all metric-learning histories contain epochs 1..{METRIC_EPOCHS}.')
print('Report figures can be regenerated from a completed run by setting HMMFF_REPORT_SOURCE_DIR.')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

box_specs = [
    (0.03, 0.56, 0.17, 0.25, 'SATURN pretraining', 'Macrogene encoder\nNo metric labels used yet', '#E8F1FA'),
    (0.29, 0.56, 0.17, 0.25, 'Label-free artifact', 'embeddings\nmacrogenes\nspecies + obs_ids', '#F3EFE6'),
    (0.80, 0.56, 0.17, 0.25, 'Post-hoc evaluation', 'Labels rejoined by obs_id\nMetrics + UMAPs', '#E9F5EA'),
]

for x, y, width, height, title, body, color in box_specs:
    patch = mpatches.FancyBboxPatch(
        (x, y), width, height,
        boxstyle='round,pad=0.018,rounding_size=0.018',
        linewidth=1.2,
        edgecolor='#333333',
        facecolor=color,
    )
    ax.add_patch(patch)
    ax.text(x + width / 2, y + height - 0.06, title, ha='center', va='top', fontsize=11, weight='bold')
    ax.text(x + width / 2, y + height / 2 - 0.03, body, ha='center', va='center', fontsize=9)

objective_x = 0.54
objective_width = 0.17
objective_y_values = [0.73, 0.56, 0.39]
objective_titles = ['InfoNCE', 'MMD-MA', 'Partial OT']
objective_bodies = ['reciprocal positives', 'distribution matching', 'transported mass']
for y, title, body in zip(objective_y_values, objective_titles, objective_bodies):
    patch = mpatches.FancyBboxPatch(
        (objective_x, y), objective_width, 0.12,
        boxstyle='round,pad=0.015,rounding_size=0.015',
        linewidth=1.0,
        edgecolor='#333333',
        facecolor='#F8F8F8',
    )
    ax.add_patch(patch)
    ax.text(objective_x + objective_width / 2, y + 0.077, title, ha='center', va='center', fontsize=10, weight='bold')
    ax.text(objective_x + objective_width / 2, y + 0.035, body, ha='center', va='center', fontsize=8)

arrow_style = dict(arrowstyle='->', lw=1.8, color='#333333', shrinkA=3, shrinkB=3)
ax.annotate('', xy=(0.29, 0.685), xytext=(0.20, 0.685), arrowprops=arrow_style)
for y in objective_y_values:
    ax.annotate('', xy=(objective_x, y + 0.06), xytext=(0.46, 0.685), arrowprops=arrow_style)
for y in objective_y_values:
    ax.annotate('', xy=(0.80, 0.685), xytext=(objective_x + objective_width, y + 0.06), arrowprops=arrow_style)

ax.text(0.54, 0.18, 'Fine-tuning sees only label-free artifact fields; cell-type labels return only for evaluation.',
        ha='center', va='center', fontsize=9, color='#444444')
fig.tight_layout()
save_report_figure(fig, 'pipeline_schematic')
plt.show()


In [ ]:
dataset_obs = load_report_truth_obs()
unknown_count = int((dataset_obs['macro_category'] == 'unknown').sum())
if unknown_count:
    unknown_labels = sorted(dataset_obs.loc[dataset_obs['macro_category'] == 'unknown', 'labels2'].unique())[:20]
    print(f'Broad categories unavailable for {unknown_count:,} cells; examples={unknown_labels}. Fine-label evaluation still includes these cells.')

species_order = pd.read_csv(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv')['species'].astype(str).tolist()
category_order = [category for category in macro_category_order if category in dataset_obs['macro_category'].unique()]
composition = (
    dataset_obs
    .groupby(['species', 'macro_category'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(index=species_order, columns=category_order, fill_value=0)
)
fine_label_counts = (
    dataset_obs
    .groupby('species', observed=True)['labels2']
    .nunique()
    .reindex(species_order)
    .reset_index(name='unique_fine_labels')
)

display(composition)
display(fine_label_counts)
print(f'Macro-category coverage: {len(dataset_obs) - unknown_count:,}/{len(dataset_obs):,} cells mapped.')


In [ ]:
source_pretrain_adata = figure_source_dir / 'baseline' / 'saturn_results' / 'adata_pretrain.h5ad'
pretrained_report = ad.read_h5ad(source_pretrain_adata)
if 'X_umap' not in pretrained_report.obsm:
    sc.pp.neighbors(pretrained_report, n_neighbors=15, metric='cosine', random_state=SEED)
    sc.tl.umap(pretrained_report, min_dist=0.3, random_state=SEED)

label_palette = make_label_palette(pretrained_report.obs['labels2'])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.umap(pretrained_report, color='species', ax=axes[0], show=False, title='Pretrained: species', size=8)
sc.pl.umap(pretrained_report, color='labels2', ax=axes[1], show=False, title='Pretrained: labels', size=8, legend_loc=None, palette=label_palette)
fig.suptitle('Shared pretrained SATURN representation', y=1.02)
fig.tight_layout()
save_report_figure(fig, 'pretrained_umap')
plt.show()


In [ ]:
history_frames = []
for trial in TRIALS:
    history = pd.read_csv(figure_source_dir / trial / 'history.csv')
    history['trial'] = trial
    history['method'] = history['trial'].map(TRIAL_DISPLAY)
    first_loss = float(history['metric_loss'].iloc[0])
    history['relative_metric_loss'] = history['metric_loss'] / first_loss
    history_frames.append(history)
histories = pd.concat(history_frames, ignore_index=True, sort=False)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
method_order = [TRIAL_DISPLAY[trial] for trial in TRIALS]

for trial in TRIALS:
    trial_history = histories[histories['trial'] == trial]
    axes[0].plot(
        trial_history['epoch'],
        trial_history['relative_metric_loss'],
        marker='o',
        ms=3,
        color=TRIAL_PALETTE[trial],
        label=TRIAL_DISPLAY[trial],
    )
axes[0].axhline(1.0, color='#444444', linestyle='--', linewidth=1)
axes[0].set(title='Objective loss relative to epoch 1', xlabel='Epoch', ylabel='Loss / epoch 1 loss')

baseline_mixing = float(summary.loc[summary['trial'] == 'baseline', 'species_mixing_fraction'].iloc[0])
for trial in LABEL_FREE_TRIALS:
    trial_history = histories[histories['trial'] == trial]
    axes[1].plot(
        trial_history['epoch'],
        trial_history['species_mixing_fraction'],
        marker='o',
        ms=3,
        color=TRIAL_PALETTE[trial],
        label=TRIAL_DISPLAY[trial],
    )
selected_rows = summary.set_index('trial')
for trial in LABEL_FREE_TRIALS:
    selected_epoch = int(selected_rows.loc[trial, 'selected_epoch'])
    selected_value = histories.loc[(histories['trial'] == trial) & (histories['epoch'] == selected_epoch), 'species_mixing_fraction']
    if len(selected_value):
        axes[1].scatter([selected_epoch], [selected_value.iloc[0]], color=TRIAL_PALETTE[trial], s=40, edgecolor='black', linewidth=0.6, zorder=4)
axes[1].axhline(baseline_mixing, color='#444444', linestyle='--', linewidth=1, label='Baseline final')
axes[1].set(title='Species integration during fine-tuning', xlabel='Epoch', ylabel='Cross-species neighbour fraction')

for trial in LABEL_FREE_TRIALS:
    trial_history = histories[histories['trial'] == trial]
    axes[2].plot(
        trial_history['epoch'],
        trial_history['teacher_top15_recall_at_50'],
        marker='o',
        ms=3,
        color=TRIAL_PALETTE[trial],
        label=TRIAL_DISPLAY[trial],
    )
axes[2].axhline(0.7, color='#444444', linestyle='--', linewidth=1, label='Checkpoint-selection floor')
axes[2].set(title='Pretrained-neighbour recall', xlabel='Epoch', ylabel='Recall@50 of pretrained top-15 neighbours', ylim=(0.65, 1.02))

for ax in axes:
    ax.legend(frameon=False, fontsize=8)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))
fig.tight_layout()
save_report_figure(fig, 'training_selection_trajectories')
save_report_figure(fig, 'loss_histories')
plt.show()


In [ ]:
metric_frame = summary.copy()
metric_frame['method'] = metric_frame['trial'].map(TRIAL_DISPLAY)
method_order = [TRIAL_DISPLAY[trial] for trial in TRIALS]
metric_labels = {
    'species_mixing_fraction': 'cross-species\nneighbours',
    'normalized_species_ilisi': 'normalized\niLISI',
    'label_same_neighbor_fraction': 'same fine\nlabel',
    'within_species_label_same_neighbor_fraction': 'within-species\nfine label',
    'species_macro_label_same_neighbor_fraction': 'macro-avg\nfine label',
    'fixed_triplet_margin_loss': 'fixed triplet\nmargin loss',
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={'width_ratios': [1.25, 1.55, 1]})

integration = metric_frame.melt(
    id_vars=['trial', 'method'],
    value_vars=['species_mixing_fraction', 'normalized_species_ilisi'],
    var_name='metric', value_name='value',
)
integration['metric_label'] = integration['metric'].map(metric_labels)
sns.barplot(data=integration, x='method', y='value', hue='metric_label', order=method_order, ax=axes[0])
axes[0].axhline(acceptance['thresholds']['species_mixing_fraction_min'], color='#444444', linestyle='--', linewidth=1)
axes[0].set(title='Species integration', xlabel=None, ylabel='Score', ylim=(0, max(0.12, integration['value'].max() * 1.25)))
axes[0].legend(title=None, frameon=False, fontsize=8)
axes[0].tick_params(axis='x', rotation=20)

preservation = metric_frame.melt(
    id_vars=['trial', 'method'],
    value_vars=[
        'label_same_neighbor_fraction',
        'within_species_label_same_neighbor_fraction',
        'species_macro_label_same_neighbor_fraction',
    ],
    var_name='metric', value_name='value',
)
preservation['metric_label'] = preservation['metric'].map(metric_labels)
sns.barplot(data=preservation, x='method', y='value', hue='metric_label', order=method_order, ax=axes[1])
axes[1].axhline(acceptance['thresholds']['label_same_neighbor_fraction_min'], color='#444444', linestyle='--', linewidth=1)
axes[1].set(title='Biological-neighbour preservation', xlabel=None, ylabel='Score', ylim=(0.68, min(1.0, preservation['value'].max() * 1.08)))
axes[1].legend(title=None, frameon=False, fontsize=8)
axes[1].tick_params(axis='x', rotation=20)

sns.barplot(data=metric_frame, x='method', y='fixed_triplet_margin_loss', order=method_order, ax=axes[2], color='#8DA0CB')
axes[2].axhline(acceptance['thresholds']['fixed_triplet_margin_loss_max'], color='#444444', linestyle='--', linewidth=1)
axes[2].set(title='Fixed triplet benchmark', xlabel=None, ylabel='Loss')
axes[2].tick_params(axis='x', rotation=20)

fig.tight_layout()
save_report_figure(fig, 'final_metric_dashboard')
save_report_figure(fig, 'mixing_metrics')
plt.show()


In [ ]:
if 'label_palette' not in globals():
    truth_for_palette = load_trial_adata('baseline')
    label_palette = make_label_palette(truth_for_palette.obs['labels2'])

fig, axes = plt.subplots(len(TRIALS), 2, figsize=(12, 20))
for row, trial in enumerate(TRIALS):
    trial_adata = load_trial_adata(trial)
    sc.pl.umap(
        trial_adata,
        color='species',
        ax=axes[row, 0],
        show=False,
        title=f'{TRIAL_DISPLAY[trial]}: species',
        size=8,
    )
    sc.pl.umap(
        trial_adata,
        color='labels2',
        ax=axes[row, 1],
        show=False,
        title=f'{TRIAL_DISPLAY[trial]}: labels',
        size=8,
        legend_loc=None,
        palette=label_palette,
    )
fig.tight_layout()
save_report_figure(fig, 'trial_umaps', dpi=300)
plt.show()


In [ ]:
cross_frames = []
preservation_frames = []
for trial in TRIALS:
    _, cross_summary, preservation_summary = compute_neighbor_diagnostics(trial, k=15)
    cross_frames.append(cross_summary)
    preservation_frames.append(preservation_summary)

cross_frame = pd.concat(cross_frames, ignore_index=True)
preservation_frame = pd.concat(preservation_frames, ignore_index=True)
cross_frame['method'] = cross_frame['trial'].map(TRIAL_DISPLAY)
preservation_frame['method'] = preservation_frame['trial'].map(TRIAL_DISPLAY)
preservation_frame['species_macro'] = preservation_frame['species'] + '\n' + preservation_frame['macro_category']

species_order = pd.read_csv(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv')['species'].astype(str).tolist()
method_order = [TRIAL_DISPLAY[trial] for trial in TRIALS]
macro_columns = [f'{species}\n{category}' for species in species_order for category in macro_category_order if category != 'unknown']

cross_pivot = (
    cross_frame
    .pivot(index='method', columns='anchor_species', values='cross_species_neighbor_fraction')
    .reindex(index=method_order, columns=species_order)
)
preservation_pivot = (
    preservation_frame
    .pivot_table(index='method', columns='species_macro', values='same_macro_neighbor_fraction')
    .reindex(index=method_order, columns=macro_columns)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [1, 2.4]})
sns.heatmap(
    cross_pivot,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    vmin=0,
    vmax=max(0.12, float(np.nanmax(cross_pivot.to_numpy())) * 1.1),
    ax=axes[0],
    cbar_kws={'label': 'Cross-species neighbour fraction'},
)
axes[0].set(title='Local species mixing by anchor species', xlabel='Anchor species', ylabel=None)

sns.heatmap(
    preservation_pivot,
    annot=False,
    cmap='Greens',
    vmin=0.5,
    vmax=1.0,
    ax=axes[1],
    cbar_kws={'label': 'Same-macro neighbour fraction'},
)
axes[1].set(title='Broad-category neighbourhood preservation', xlabel='Species / macro category', ylabel=None)
axes[1].tick_params(axis='x', rotation=65)

fig.tight_layout()
save_report_figure(fig, 'local_neighbor_diagnostics')
plt.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if 'label_palette' not in globals():
    truth_for_palette = load_trial_adata('baseline')
    label_palette = make_label_palette(truth_for_palette.obs['labels2'])

interactive_trials = ['baseline', 'infonce']
fig = make_subplots(
    rows=1,
    cols=len(interactive_trials),
    subplot_titles=[TRIAL_DISPLAY[trial] for trial in interactive_trials],
    horizontal_spacing=0.04,
)

for col, trial in enumerate(interactive_trials, start=1):
    trial_adata = load_trial_adata(trial)
    hover_frame = pd.DataFrame({
        'umap_1': trial_adata.obsm['X_umap'][:, 0],
        'umap_2': trial_adata.obsm['X_umap'][:, 1],
        'obs_id': trial_adata.obs_names.astype(str),
        'species': trial_adata.obs['species'].astype(str).to_numpy(),
        'cell_type': trial_adata.obs['labels2'].astype(str).to_numpy(),
    })
    point_colors = hover_frame['cell_type'].map(label_palette).fillna('#808080')
    fig.add_trace(
        go.Scattergl(
            x=hover_frame['umap_1'],
            y=hover_frame['umap_2'],
            mode='markers',
            marker={'size': 3, 'color': point_colors, 'opacity': 0.75},
            customdata=hover_frame[['obs_id', 'species', 'cell_type']].to_numpy(),
            hovertemplate=(
                'obs_id=%{customdata[0]}<br>'
                'species=%{customdata[1]}<br>'
                'cell_type=%{customdata[2]}'
                f'<extra>{TRIAL_DISPLAY[trial]}</extra>'
            ),
            showlegend=False,
        ),
        row=1,
        col=col,
    )

fig.update_layout(
    title='Interactive UMAP hover: baseline vs InfoNCE',
    template='plotly_white',
    width=1150,
    height=560,
    margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
)
for col in range(1, len(interactive_trials) + 1):
    fig.update_xaxes(title_text='UMAP 1', showgrid=False, zeroline=False, row=1, col=col)
    fig.update_yaxes(title_text='UMAP 2', showgrid=False, zeroline=False, row=1, col=col)
fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plot_category_order = list(macro_category_order)
interactive_trials = ['baseline', 'infonce']
fig = make_subplots(
    rows=1,
    cols=len(interactive_trials),
    subplot_titles=[TRIAL_DISPLAY[trial] for trial in interactive_trials],
    horizontal_spacing=0.04,
)

for col, trial in enumerate(interactive_trials, start=1):
    trial_adata = load_trial_adata(trial)
    hover_frame = pd.DataFrame({
        'umap_1': trial_adata.obsm['X_umap'][:, 0],
        'umap_2': trial_adata.obsm['X_umap'][:, 1],
        'obs_id': trial_adata.obs_names.astype(str),
        'species': trial_adata.obs['species'].astype(str).to_numpy(),
        'cell_type': trial_adata.obs['labels2'].astype(str).to_numpy(),
    })
    hover_frame['macro_category'] = macro_categories_for(hover_frame['species'], hover_frame['cell_type'])

    for category in plot_category_order:
        category_frame = hover_frame[hover_frame['macro_category'] == category]
        if category_frame.empty:
            continue
        fig.add_trace(
            go.Scattergl(
                x=category_frame['umap_1'],
                y=category_frame['umap_2'],
                mode='markers',
                marker={'size': 2, 'color': macro_category_palette[category], 'opacity': 0.75},
                customdata=category_frame[['obs_id', 'species', 'cell_type', 'macro_category']].to_numpy(),
                hovertemplate=(
                    'obs_id=%{customdata[0]}<br>'
                    'species=%{customdata[1]}<br>'
                    'macro_category=%{customdata[3]}<br>'
                    'cell_type=%{customdata[2]}'
                    f'<extra>{TRIAL_DISPLAY[trial]}</extra>'
                ),
                name=category,
                legendgroup=category,
                showlegend=(col == 1),
            ),
            row=1,
            col=col,
        )

fig.update_layout(
    title='Interactive UMAP hover: baseline vs InfoNCE macro categories',
    template='plotly_white',
    width=1150,
    height=560,
    margin={'l': 20, 'r': 20, 't': 70, 'b': 20},
    legend_title_text='Macro category',
)
for col in range(1, len(interactive_trials) + 1):
    fig.update_xaxes(title_text='UMAP 1', showgrid=False, zeroline=False, row=1, col=col)
    fig.update_yaxes(title_text='UMAP 2', showgrid=False, zeroline=False, row=1, col=col)
fig.show()


In [ ]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#HIGHLIGHT_FINE_LABEL = 'plasma'
HIGHLIGHT_FINE_LABEL = 'venous'

MATCH_MODE = 'exact'  # use 'exact' to require an exact labels2 value
WRITE_HTML = False

required_helpers = ['load_trial_adata', 'TRIAL_DISPLAY', 'repo_root']
missing_helpers = [name for name in required_helpers if name not in globals()]
if missing_helpers:
    raise RuntimeError(f'Run the section 6 report helper cell first; missing {missing_helpers}.')

query = str(HIGHLIGHT_FINE_LABEL).strip()
if not query:
    raise ValueError('Set HIGHLIGHT_FINE_LABEL to a non-empty labels2 value or substring.')
query_norm = query.casefold()

def fine_label_mask(labels, mode):
    values = pd.Series(labels).astype(str)
    normalized = values.str.casefold()
    if mode == 'contains':
        return normalized.str.contains(re.escape(query_norm), na=False).to_numpy()
    if mode == 'exact':
        return (normalized == query_norm).to_numpy()
    raise ValueError("MATCH_MODE must be either 'contains' or 'exact'.")

highlight_trials = ['baseline', 'infonce']
species_order = pd.read_csv(repo_root / 'data' / 'human_monkey_mouse_frog_fish.csv')['species'].astype(str).tolist()
species_palette = dict(zip(species_order, sns.color_palette('Set1', n_colors=len(species_order)).as_hex()))

trial_frames = {}
count_rows = []
for trial in highlight_trials:
    trial_adata = load_trial_adata(trial)
    frame = pd.DataFrame({
        'umap_1': trial_adata.obsm['X_umap'][:, 0],
        'umap_2': trial_adata.obsm['X_umap'][:, 1],
        'obs_id': trial_adata.obs_names.astype(str),
        'species': trial_adata.obs['species'].astype(str).to_numpy(),
        'cell_type': trial_adata.obs['labels2'].astype(str).to_numpy(),
    })
    frame['highlight'] = fine_label_mask(frame['cell_type'], MATCH_MODE)
    trial_frames[trial] = frame
    for species in species_order:
        count_rows.append({
            'method': TRIAL_DISPLAY[trial],
            'species': species,
            'highlighted_cells': int(((frame['highlight']) & (frame['species'] == species)).sum()),
        })

if not any(frame['highlight'].any() for frame in trial_frames.values()):
    available_labels = sorted(trial_frames['baseline']['cell_type'].unique())
    suggestions = [label for label in available_labels if query_norm in label.casefold()][:20]
    raise ValueError(
        f'No labels2 values matched {query!r} with MATCH_MODE={MATCH_MODE!r}. '
        f'Containing-label examples: {suggestions}'
    )

count_frame = pd.DataFrame(count_rows)
display(count_frame.pivot(index='species', columns='method', values='highlighted_cells'))

subplot_titles = [
    f"{TRIAL_DISPLAY[trial]} (n={int(trial_frames[trial]['highlight'].sum()):,})"
    for trial in highlight_trials
]
fig = make_subplots(
    rows=1,
    cols=len(highlight_trials),
    subplot_titles=subplot_titles,
    horizontal_spacing=0.04,
)

for col, trial in enumerate(highlight_trials, start=1):
    frame = trial_frames[trial]
    background = frame[~frame['highlight']]
    fig.add_trace(
        go.Scattergl(
            x=background['umap_1'],
            y=background['umap_2'],
            mode='markers',
            marker={'size': 2, 'color': '#D0D0D0', 'opacity': 0.18},
            hoverinfo='skip',
            showlegend=False,
        ),
        row=1,
        col=col,
    )

    for species in species_order:
        species_frame = frame[(frame['highlight']) & (frame['species'] == species)]
        if species_frame.empty:
            continue
        fig.add_trace(
            go.Scattergl(
                x=species_frame['umap_1'],
                y=species_frame['umap_2'],
                mode='markers',
                marker={
                    'size': 5,
                    'color': species_palette.get(species, '#333333'),
                    'opacity': 0.9,
                    'line': {'width': 0.35, 'color': '#222222'},
                },
                customdata=species_frame[['obs_id', 'species', 'cell_type']].to_numpy(),
                hovertemplate=(
                    'obs_id=%{customdata[0]}<br>'
                    'species=%{customdata[1]}<br>'
                    'cell_type=%{customdata[2]}'
                    f'<extra>{TRIAL_DISPLAY[trial]}</extra>'
                ),
                name=species,
                legendgroup=species,
                showlegend=(col == 1),
            ),
            row=1,
            col=col,
        )

fig.update_layout(
    title=f"Baseline vs InfoNCE: highlighted labels2 {MATCH_MODE} {query!r}",
    template='plotly_white',
    width=1150,
    height=560,
    margin={'l': 20, 'r': 20, 't': 80, 'b': 20},
    legend_title_text='Highlighted species',
)
for col in range(1, len(highlight_trials) + 1):
    fig.update_xaxes(title_text='UMAP 1', showgrid=False, zeroline=False, row=1, col=col)
    fig.update_yaxes(title_text='UMAP 2', showgrid=False, zeroline=False, row=1, col=col)

if WRITE_HTML:
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', query).strip('_') or 'highlight'
    html_path = assets / f'baseline_vs_infonce_{slug}_highlight.html'
    fig.write_html(html_path)
    print(f'Saved {relative_report_path(html_path)}')

fig.show()
